In [1]:
from pathlib import Path
import pandas as pd
import re
import unicodedata

DATA_DIR = Path('../training_datasets')

S1_PATH = DATA_DIR / 'train_source1.tsv'
S2_PATH = DATA_DIR / 'train_source2.tsv'
S3_PATH = DATA_DIR / 'train_source3.tsv'
GT_PATH = DATA_DIR / 'train_ground_truth.tsv'

CHUNK_SIZE = 100_000

# Business Name Normalization

In [10]:
HONORIFICS = {'mr', 'mrs', 'ms', 'dr'}          # sri / smt removed

COMPOUND_SUFFIXES = {
    ('private', 'limited'),
    ('pvt', 'ltd'),
    ('pvt', 'limited'),
}

SUFFIXES = {
    'inc', 'incorporated',
    'ltd', 'limited',
    'llc',
    'corp', 'corporation',
    'co', 'company',
    'pvt',
    'llp',
    'plc'
}

DOTTED_SUFFIX_MAP = {
    r'\bl\.?\s*l\.?\s*c\.?\b': 'llc',
    r'\bp\.?\s*l\.?\s*c\.?\b': 'plc',
    r'\bl\.?\s*t\.?\s*d\.?\b': 'ltd',
    r'\binc\.?\b': 'inc',
    r'\bcorp\.?\b': 'corp',
    r'\bco\.?\b': 'co',
}

def normalize_name(name: str) -> str:
    if not isinstance(name, str) or not name.strip():
        return ''

    # 1. NFKC
    s = unicodedata.normalize('NFKC', name)

    # 2. lowercase
    s = s.lower()

    # 3. dotted legal forms → canonical
    for pattern, replacement in DOTTED_SUFFIX_MAP.items():
        s = re.sub(pattern, replacement, s)

    # 4. & → and (track whether we did it)
    ampersand_replaced = '&' in s
    s = s.replace('&', ' and ')

    # 5. punctuation → spaces
    s = re.sub(r'[^\w\s]', ' ', s)

    # 6. collapse whitespace
    s = re.sub(r'\s+', ' ', s).strip()

    tokens = s.split()

    # 7. remove leading honorifics
    while tokens and tokens[0] in HONORIFICS:
        tokens.pop(0)

    # 8. remove compound legal suffixes first
    for compound in COMPOUND_SUFFIXES:
        n = len(compound)
        if len(tokens) >= n and tuple(tokens[-n:]) == compound:
            tokens = tokens[:-n]
            break

    # 9. remove ordinary trailing suffixes
    while tokens and tokens[-1] in SUFFIXES:
        tokens.pop()

    # 10. remove dangling "and" only if we created it
    if ampersand_replaced and tokens and tokens[-1] == 'and':
        tokens.pop()

    return ' '.join(tokens)

In [11]:
test_names = [
    "Dr. Sharma & Co.",
    "Sharma & Company",
    "SRI RAM PRIVATE LIMITED",
    "SRI RAM PVT LTD",
    "Tata Motors Ltd.",
    "Apple Inc.",
    "Microsoft Corporation",
    "ABC L.L.C.",
    "ABC L T D",
    "ABC P.L.C.",
    "MÜLLER & SÖHNE",
    "Dr.   Sharma   &   Co.",
    "Sharma and Sons",
    "Sri Lakshmi Traders",
]

for name in test_names:
    print(f"{name!r:40s} → {normalize_name(name)!r}")

'Dr. Sharma & Co.'                       → 'sharma'
'Sharma & Company'                       → 'sharma'
'SRI RAM PRIVATE LIMITED'                → 'sri ram'
'SRI RAM PVT LTD'                        → 'sri ram'
'Tata Motors Ltd.'                       → 'tata motors'
'Apple Inc.'                             → 'apple'
'Microsoft Corporation'                  → 'microsoft'
'ABC L.L.C.'                             → 'abc'
'ABC L T D'                              → 'abc'
'ABC P.L.C.'                             → 'abc'
'MÜLLER & SÖHNE'                         → 'müller and söhne'
'Dr.   Sharma   &   Co.'                 → 'sharma'
'Sharma and Sons'                        → 'sharma and sons'
'Sri Lakshmi Traders'                    → 'sri lakshmi traders'
